教学文件和生成内容保存在 /root/gpufree-data/PATHAK/tutorials/。选择 Python (pathak) 内核，按顺序执行。

PROJECT_ROOT 指向项目根目录，TUTORIALS_ROOT 指向教学目录。BioBERT 完整模型位于 models/biobert_pubmed_sentence/，Java JAR 和分句资源位于 resources/。

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("/root/gpufree-data/PATHAK")
TUTORIALS_ROOT = PROJECT_ROOT / "tutorials"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import importlib
import pathak_runtime
importlib.reload(pathak_runtime)
from pathak_runtime import configure
configure()
print("Project:", PROJECT_ROOT)
print("Tutorials:", TUTORIALS_ROOT)
print("Python:", sys.executable)


# GO Terms Defination Embedding
<div>
<img src="https://www.johnsnowlabs.com/wp-content/uploads/2019/07/johnsnowlabs_logo.png" width="200"/>
</div>

**NLU**:State of the Art Text Mining in Python
__The Simplicity of Python, the Power of Spark NLP__



::::{grid}
:gutter: 4

:::{grid-item-card} Input File
**GO_BP_Ontology_corups.tsv**, 
:::

:::{grid-item-card} Input Folder
**None**
:::

:::{grid-item-card} Output File
**GO_BP_Ontology_corups_Filtered.tsv**, 
**predictions_full_BP.pickle**
:::

:::{grid-item-card} Output Folder
**pipe_full_BP**
:::
::::

<div class="alert alert-block alert-info">
    <h2>🐍Load Python libraries</h2>
</div>

In [2]:
import os
import nlu
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pickle
from IPython.display import display, HTML

<div class="alert alert-block alert-warning">
    <h2>Make sure java is loaded</h2>
</div>

In [3]:
!java -version

openjdk version "1.8.0_472"
OpenJDK Runtime Environment (Zulu 8.90.0.19-CA-linux64) (build 1.8.0_472-b08)
OpenJDK 64-Bit Server VM (Zulu 8.90.0.19-CA-linux64) (build 25.472-b08, mixed mode)


<div class="alert alert-block alert-info">
    <h2>Set Java JAVA_HOME and PATH variable</h2>
</div>

In [4]:
print("Java home:", os.environ["JAVA_HOME"])

Java home: /opt/conda/envs/pathak


In [5]:
print(os.environ["JAVA_HOME"])

/opt/conda/envs/pathak


***
<div class="alert alert-block alert-info">
Load ontology file
</div>

In [ ]:
df = pd.read_csv(TUTORIALS_ROOT / "GO_BP_Ontology_corups.tsv", sep="\t")  # GO 语料
print(df.shape)
display(df.head())

<div class="alert alert-block alert-warning">
    <h2>Filter GO terms which are not in TAIR database to focus only on the Plant associated GO terms</h2>
</div>

In [ ]:
TAIR_terms = open(TUTORIALS_ROOT / "Auxiliary_Files/TAIR_GOTERMS.txt").read().splitlines()
len(TAIR_terms)

Save filter GO term data

In [ ]:
df = df[df.GO.isin(TAIR_terms)]
df.to_csv(TUTORIALS_ROOT / "GO_BP_Ontology_corups_Filtered.tsv", index=False, sep="\t")  # 筛选 GO 语料
df.shape

加载本地 BioBERT 句向量模型。

完整目录为 /root/gpufree-data/PATHAK/models/biobert_pubmed_sentence/，其中 bert_sentence_tensorflow 是模型文件，fields/vocabulary/ 是词表，metadata/ 是加载配置。

本课使用预训练模型生成向量。predictions_full_BP.pickle 和 pipe_full_BP/ 都写入 tutorials/，供第三课读取。


In [9]:
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

<div class="alert alert-block alert-info">
    <h2>Load BioBERT Sentence Embeddings (Pubmed)</h2>
    This model contains a pre-trained weights of BioBERT, a language representation model for biomedical domain, especially designed for biomedical text mining tasks such as biomedical named entity recognition, relation extraction, question answering, etc. 
</div>

The details are described in the paper [“BioBERT: a pre-trained biomedical language representation model for biomedical text mining”](https://arxiv.org/abs/1901.08746).

In [ ]:
%%time
from pathak_runtime import load_biobert, BIOBERT_PATH, BIOBERT_TF_CONFIG, BIOBERT_BATCH_SIZE

# Spark NLP 格式的完整本地模型：模型文件、词表和配置一起保存。
MODEL_PATH = BIOBERT_PATH
print("Local BioBERT model:", MODEL_PATH)
# 首次使用 CPU8 配置前重启内核；全局计算池由 JVM 的第一个 TF Session 确定。
print("TF config / batch:", BIOBERT_TF_CONFIG, BIOBERT_BATCH_SIZE)
pipe = load_biobert(model_path=MODEL_PATH)


<div class="alert alert-block alert-info">
    <h2>Get the sentence embedding (biobert) of GO term definition</h2>
</div>

The details are described in the paper __[“BioBERT: a pre-trained biomedical language representation model for biomedical text mining”](https://arxiv.org/abs/1901.08746)__.

In [ ]:
# 默认 batchSize=16。TensorFlow 图内并行逐句编码，共享权重并保留原单句计算顺序。
# 图内按 input_mask 排除 PAD；保留 CLS/SEP，仍返回每条定义一个 768 维向量。
predictions_full = pipe.predict(df.Definition, output_level='document')  # 一条英文定义使用 BioBERT 编码为一个包含 768 个数值的向量
predictions_full
# 实际输入模型的是 Definition 列。 GO 编号、名称和 Depth 在这一步用于保留术语信息，并没有作为这次编码的输入特征。

<div class="alert alert-block alert-info">
    <h2>Save as embeddings as Pickle</h2>
</div>

In [ ]:
%%time

with open(TUTORIALS_ROOT / "predictions_full_BP.pickle", 'wb') as handle:
    pickle.dump(predictions_full, handle, protocol=pickle.HIGHEST_PROTOCOL)
# pickle是一种序列化格式。'wb'是以二进制形式打开文件。
# with open(TUTORIALS_ROOT / "predictions_full_bkp.pickle", 'wb') as handle:
#     pickle.dump(predictions_full, handle, protocol=pickle.HIGHEST_PROTOCOL)
    
# !ls *.pickle

<div class="alert alert-block alert-info">
    <h2>Save as Pipe</h2>
</div>

In [ ]:
%%time
stored_model_path = str(TUTORIALS_ROOT / "pipe_full_BP")
pipe.save(stored_model_path, overwrite=True)

from pathak_runtime import write_embedding_manifest
embedding_manifest = write_embedding_manifest(
    pipe,
    TUTORIALS_ROOT / "GO_BP_Ontology_corups_Filtered.tsv",
    TUTORIALS_ROOT / "predictions_full_BP.pickle",
    TUTORIALS_ROOT / "pipe_full_BP",
)
embedding_manifest


### <b>(Optional): </b>Load Saved model

```
with open('predictions_full_BP.pickle', 'rb') as handle:
    predictions = pickle.load(handle)
predictions

```